# Actividad 3 — Data Wrangling: Room Occupancy Estimation

**Nombre:** Luis Manuel Velasco Iglesias
**Matrícula:** A00226599
**Curso:** TC5044.10 MLOps

**Objetivo.** `modified.csv` es una copia perturbada del dataset *Room Occupancy Estimation* (UCI).
En este notebook se diagnostica su calidad, se limpia con dos estrategias y se compara cada
resultado contra el dataset original sin duplicados.

**Regla de trabajo.** Todas las decisiones de limpieza se toman solo con `modified.csv` y
conocimiento del dominio. El archivo `original.csv` se usa únicamente al final, para la
auditoría y la EDA comparativa.

**Estructura de carpetas esperada** (rutas relativas a este notebook):

    notebooks/A03_A00226599.ipynb
    data/raw/room-occupancy-estimation/original.csv
    data/raw/room-occupancy-estimation/modified.csv
    data/raw/room-occupancy-estimation/metadata.json
    data/interim/      <- línea base sin duplicados (se genera aquí)
    data/processed/    <- un archivo limpio por estrategia (se genera aquí)

### 1. Configuracion 

In [ ]:
# ------ Import Libraries ----------

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [2]:
# ------ File Paths ----------

# Todas las rutas se definen aquí. Si la estructura cambia, solo se edita PROJECT_DIR.
PROJECT_DIR = Path("..")  # el notebook vive en notebooks/
RAW_DIR = PROJECT_DIR / "data" / "raw" / "room-occupancy-estimation"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"

MODIFIED_PATH = RAW_DIR / "modified.csv"
ORIGINAL_PATH = RAW_DIR / "original.csv"        # solo para auditoría y EDA comparativa
METADATA_PATH = RAW_DIR / "metadata.json"
BASELINE_PATH = INTERIM_DIR / "original_no_duplicates.csv"
CLEAN_A_PATH = PROCESSED_DIR / "cleaned_dropna.csv"    # estrategia A
CLEAN_B_PATH = PROCESSED_DIR / "cleaned_imputed.csv"   # estrategia B

# Crear las carpetas de salida y fallar pronto si faltan los datos de entrada
for folder in (INTERIM_DIR, PROCESSED_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

for path in (MODIFIED_PATH, ORIGINAL_PATH):
    if not path.exists():
        raise FileNotFoundError(f"No se encontró {path}. Revisa la estructura de carpetas descrita al inicio.")

### 2. Carga segura de los datos

El archivo se lee completo como texto (`dtype=str`). Si pandas infiere los tipos, un solo
token como `"?"` convierte la columna entera a `object` y los problemas quedan ocultos.
Con `keep_default_na=False`, cadenas como `"NULL"` o `"N/A"` llegan como texto literal,
de modo que las pruebas de diagnóstico puedan contarlas en lugar de que desaparezcan como NaN.
python

In [3]:
raw_df = pd.read_csv(MODIFIED_PATH, dtype=str, keep_default_na=False)

print(f"Filas: {raw_df.shape[0]:,}   Columnas: {raw_df.shape[1]}")
raw_df.head()

Filas: 10,331   Columnas: 20


,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count,mixed_type_col
0,2017/12/22,10:49:41,24.94,345.75,24.56,25.38,121.0,34.0,53.0,40.0,0.08,0.19,0.06,0.06,390.0,0.769230769231,0.0,0.0,1.0,599
1,2017/12/22,10:50:12,24.94,24.75,24.56,25.44,121.0,33.0,53.0,40.0,0.93,0.05,0.06,0.06,390.0,0.646153846154,0.0,0.0,1.0,
2,2017/12/22,10:50:42,25.0,24.75,24.5,25.44,121.0,34.0,53.0,40.0,0.43,0.11,0.08,0.06,390.0,0.519230769231,0.0,0.0,1.0,unknown
3,2017/12/22,10:51:13,25.0,24.75,24.56,25.44,121.0,34.0,53.0,40.0,0.41,0.1,0.1,0.09,390.0,0.388461538462,0.0,0.0,1.0,584
4,2017/12/22,10:51:44,25.0,24.75,24.56,25.44,121.0,34.0,54.0,40.0,0.18,0.06,0.06,0.06,390.0,0.253846153846,0.0,0.0,1.0,bad


In [4]:
# Metadatos de la perturbación: solo como referencia, no se usan para decidir la limpieza
metadata = json.loads(METADATA_PATH.read_text())
metadata["rates"]

{'dup_rate': 0.02,
 'nan_rate': 0.01,
 'outlier_rate': 0.01,
 'bad_token_rate': 0.005,
 'obj_noise_rate': 0.05,
 'add_mixed_type': True}